<a href="https://colab.research.google.com/github/smriti3003/Agents/blob/main/AI_Agent_Assignment_1_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Assignment 1.2 — AI Agent with LlamaIndex & Google Gemini

This notebook follows the structure of the provided lab notebook and extends the weather-agent example to include **3 sensors and 3 actuators**.

**Sensors:** weather, Wikipedia knowledge, current date/time.

**Actuators:** save recommendation to TXT, save structured result to JSON, append an interaction log to CSV.

> The code is written for Google Colab. The notebook is intentionally not executed here so the Gemini API key is not exposed.

In [1]:
%pip install -qU "google-genai>=2.9.0"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 16.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 259.1/259.1 kB 11.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.56.3 which is incompatible.


In [2]:
from google import genai
from google.colab import userdata

GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
client = genai.Client(api_key=GEMINI_API_KEY)

In [6]:
import os
!pip install -q llama-index llama-index-llms-google-genai requests wikipedia langchain


## 1. Connect the Gemini API

Before running this cell, create a Colab Secret named `GOOGLE_API_KEY` and paste your Gemini API key into it.

In [4]:
# @title
!pip install -q llama-index llama-index-llms-google-genai requests

import os
import requests

from llama_index.llms.google_genai import GoogleGenAI
from llama_index.core.agent.workflow import FunctionAgent
from llama_index.core.tools import FunctionTool



# Read the Gemini key from Colab Secrets.
# Do NOT paste the key directly into the notebook.
GOOGLE_API_KEY = userdata.get('GEMINI_API_KEY')

if not GOOGLE_API_KEY:
    raise ValueError("GOOGLE_API_KEY was not found in Colab Secrets.")

os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
print("Gemini API key loaded successfully.")


Gemini API key loaded successfully.


## 2. Import LlamaIndex and supporting libraries

In [8]:
import csv
import json
import requests
from datetime import datetime

from llama_index.llms.google_genai import GoogleGenAI
from llama_index.core.agent.workflow import FunctionAgent
from llama_index.core.tools import FunctionTool




## 3. Gemini model

The lab example uses Gemini through LlamaIndex. The model below can be changed if your course account/API currently exposes a different Gemini model.

In [9]:
# -------------------------------------
# Gemini Model
# -------------------------------------

llm = GoogleGenAI(
    model="gemini-3.5-flash",
    api_key=GEMINI_API_KEY
)

# 4. Sensors


### Sensor 1 — Weather
Gets current temperature, humidity, and weather description from wttr.in.

### Sensor 2 — Wikipedia
Gets background knowledge about a topic.

### Sensor 3 — Current date/time
Gets the current timestamp so the recommendation can be time-aware.

In [11]:
# -------------------------------------
# SENSOR 1
# -------------------------------------

def get_weather(city: str) -> str:

    url = f"https://wttr.in/{city}?format=j1"

    response = requests.get(url)

    if response.status_code != 200:
        return "Unable to retrieve weather information."

    data = response.json()

    current_weather = data["current_condition"][0]

    temperature = current_weather["temp_C"]
    humidity = current_weather["humidity"]
    description = current_weather["weatherDesc"][0]["value"]

    return f"""
    City: {city}
    Temperature: {temperature}°C
    Humidity: {humidity}%
    Weather: {description}
    """


# -------------------------------------
# ACTUATOR
# -------------------------------------

def save_recommendation(recommendation: str) -> str:

    with open("weather_recommendation.txt", "w") as file:
        file.write(recommendation)

    return "Recommendation successfully saved."


# -------------------------------------
# CREATE TOOLS
# -------------------------------------

weather_sensor = FunctionTool.from_defaults(
    fn=get_weather
)

save_actuator = FunctionTool.from_defaults(
    fn=save_recommendation
)


# -------------------------------------
# CREATE AGENT
# -------------------------------------

agent = FunctionAgent(
    llm=llm,
    tools=[
        weather_sensor,
        save_actuator
    ],
    system_prompt="""
    You are a weather recommendation agent.

    Use the weather sensor to obtain current weather information.
    Analyze the information.
    Recommend appropriate clothing and whether an umbrella is needed.
    Finally, save your recommendation using the save_recommendation tool.
    """
)


# -------------------------------------
# SENSOR 2 — Wikipedia
# -------------------------------------

def search_wikipedia(topic: str) -> str:
    """Retrieve a short Wikipedia summary for a topic."""

    url = "https://en.wikipedia.org/api/rest_v1/page/summary/" + topic.replace(" ", "_")

    response = requests.get(url, timeout=10)

    if response.status_code != 200:
        return f"Unable to retrieve Wikipedia information for {topic}."

    data = response.json()

    title = data.get("title", topic)
    extract = data.get("extract", "No summary available.")

    return f"""
Topic: {title}

Wikipedia Summary:
{extract}
"""

wikipedia_sensor = FunctionTool.from_defaults(
    fn=search_wikipedia
)

# -------------------------------------
# SENSOR 3 — Current date/time
# -------------------------------------
def get_current_datetime() -> str:
    """Return the current local system date and time."""
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")


# 5. Actuators

An actuator is a tool that allows the agent to take an action in the environment.

### Actuator 1 — Save a human-readable recommendation
### Actuator 2 — Save a structured JSON result
### Actuator 3 — Append the interaction to a CSV log

In [12]:
# -------------------------------------
# ACTUATOR 1 — Save TXT recommendation
# -------------------------------------
def save_recommendation(recommendation: str) -> str:
    with open("weather_recommendation.txt", "w", encoding="utf-8") as file:
        file.write(recommendation)

    return "Recommendation successfully saved to weather_recommendation.txt."


# -------------------------------------
# ACTUATOR 2 — Save JSON result
# -------------------------------------
def save_json_result(city: str, recommendation: str) -> str:
    result = {
        "city": city,
        "recommendation": recommendation,
        "saved_at": datetime.now().isoformat()
    }

    with open("weather_result.json", "w", encoding="utf-8") as file:
        json.dump(result, file, indent=2)

    return "Structured result successfully saved to weather_result.json."


# -------------------------------------
# ACTUATOR 3 — Append interaction log
# -------------------------------------
def log_interaction(city: str, recommendation: str) -> str:
    file_exists = os.path.exists("agent_history.csv")

    with open("agent_history.csv", "a", newline="", encoding="utf-8") as file:
        writer = csv.writer(file)

        if not file_exists:
            writer.writerow(["timestamp", "city", "recommendation"])

        writer.writerow([
            datetime.now().isoformat(),
            city,
            recommendation
        ])

    return "Interaction successfully logged to agent_history.csv."


# 6. Convert functions into LlamaIndex tools

In [13]:
weather_sensor = FunctionTool.from_defaults(
    fn=get_weather,
    name="weather_sensor",
    description="Gets current temperature, humidity, and weather conditions for a city."
)

datetime_sensor = FunctionTool.from_defaults(
    fn=get_current_datetime,
    name="datetime_sensor",
    description="Gets the current date and time."
)

save_actuator = FunctionTool.from_defaults(
    fn=save_recommendation,
    name="save_recommendation",
    description="Saves the final recommendation to a text file."
)

json_actuator = FunctionTool.from_defaults(
    fn=save_json_result,
    name="save_json_result",
    description="Saves the city and recommendation as a structured JSON file."
)

log_actuator = FunctionTool.from_defaults(
    fn=log_interaction,
    name="log_interaction",
    description="Appends the city, timestamp, and recommendation to a CSV history file."
)


# 7. Create the AI Agent

The system prompt explicitly tells the agent when to use the sensors and actuators.

In [14]:
agent = FunctionAgent(
    llm=llm,
    tools=[
    weather_sensor,
    wikipedia_sensor,
    datetime_sensor,
    save_actuator,
    json_actuator,
    log_actuator
      ],
    system_prompt="""
You are a helpful weather and travel recommendation agent.

For a city-based request:
1. Use the weather sensor to obtain current weather information.
2. Use the date/time sensor when timing is relevant.
3. Use the Wikipedia sensor when the user asks for background information about a place or topic.
4. Analyze the information using your reasoning capabilities.
5. Give a concise recommendation about clothing, umbrella/rain protection, and practical preparation.
6. When the user asks to save the result, use the save_recommendation actuator.
7. When appropriate, save a structured result using save_json_result.
8. Log the interaction using log_interaction.

Do not invent sensor readings. If a sensor fails, clearly state that the information could not be retrieved.
"""
)


# 8. Run the agent

This cell is intentionally written asynchronously, matching the `await agent.run(...)` pattern from the provided lab example.

In [15]:
try:
    response = await agent.run(
        "Check the current weather in Hyderabad. "
        "Tell me what I should wear and whether I should carry an umbrella. "
        "Also give me one useful fact about Hyderabad from Wikipedia. "
        "Save the recommendation to a text file, save a JSON result, "
        "and log the interaction to the CSV history file."
    )

    print(response)

except Exception as e:
    import traceback
    traceback.print_exc()


### Weather and Recommendations for Hyderabad

*   **Current Temperature:** 28°C
*   **Humidity:** 63%
*   **Weather Condition:** Patchy rain nearby

#### Recommendations:
*   **What to Wear:** Opt for light, breathable, and comfortable cotton clothing due to the warm temperatures and moderate humidity.
*   **Umbrella/Rain Protection:** Yes, **carry an umbrella** or a light raincoat with you, as there is patchy rain in the vicinity and you might encounter sudden showers.

---

### Wikipedia Fact
*Note: I attempted to retrieve background information for Hyderabad, Hyderabad (India), and the Charminar from Wikipedia, but the system returned an error indicating that the information could not be retrieved at this time.*

---

### File and Log Actions Completed:
1.  **Text File Saved:** The recommendation has been saved to `weather_recommendation.txt`.
2.  **JSON Result Saved:** A structured representation has been saved to `weather_result.json`.
3.  **Interaction Logged:** The transaction 

# 9. Inspect the actuator outputs

After a successful run, these files should appear in the Colab working directory:

- `weather_recommendation.txt`
- `weather_result.json`
- `agent_history.csv`


In [16]:
import os

for filename in [
    "weather_recommendation.txt",
    "weather_result.json",
    "agent_history.csv"
]:
    print(filename, "->", "created" if os.path.exists(filename) else "not created")


weather_recommendation.txt -> created
weather_result.json -> created
agent_history.csv -> created
